<h1> Parsing + Chunking </h1>

In [ ]:
from unstructured.partition.pdf import partition_pdf #pdf files

from unstructured.chunking.title import chunk_by_title #https://docs.unstructured.io/open-source/core-functionality/chunking

from pathlib import Path
from langchain_core.documents import Document

In [ ]:
path = Path (r"C:\Users\Admin\Desktop\ip\How To RAG\docs")

documentos = []
chunk_id = 1

for docs in path.iterdir():

    file_dtype = docs.suffix.lower()

    if file_dtype == ".pdf":

        parsing = partition_pdf (str(docs), languages = ["por"])
        chunks = chunk_by_title (parsing[3:], max_characters = 750)

    '''elif file_dtype == ".txt":

        parsing = partition_text (str(docs), languages = ["por"])
        chunks = chunk_elements (parsing, max_characters = 750)

    elif file_dtype == ".docx":        
        
        parsing = partition_docx (str(docs), languages = ["por"])
        chunks = chunk_by_title (parsing, max_characters = 750)'''


    for chunk in chunks:

        documentos.append (
            Document (
                page_content = chunk.text,
                metadata = {
                    "title": docs.name,
                    "chunk_id": chunk_id
                }
            )
        )

        chunk_id += 1


#documentos

In [ ]:
display (documentos)

'''for doc in documentos:
    print (doc.page_content)
'''

<hr>

<h1> Retrieval</h1>

<h3> Sparse Retrieval </h3>

In [ ]:
from langchain_community.retrievers import BM25Retriever #https://reference.langchain.com/python/langchain-community/retrievers

sparse_retrieval = BM25Retriever.from_documents (documentos, k = 50) #@K aqui definido

<hr>

<h3> Dense Retrieval </h3>

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings #https://reference.langchain.com/python/langchain-huggingface
from langchain_community.vectorstores import FAISS #https://docs.langchain.com/oss/python/integrations/vectorstores #Não o melhor para produção


emb_hf = HuggingFaceEmbeddings (model_name = r"C:\Users\Admin\Desktop\models\Jina") #Embeddings

db = FAISS.from_documents (documentos, emb_hf) #Indexação

In [ ]:
dense_retrieval = db.as_retriever (
    search_type = "similarity", #similarity, mmr, similarity_score_threshold
    search_kwargs = {
        "k": 50
    }
)

<hr>

<h3> Eval Retrieval </h3>

In [ ]:
#Dataset
import json

with open (r"C:\Users\Admin\Desktop\ip\Synthetic Data\dataset-Mistral.json", "r", encoding = "utf-8") as f:
    dataset = json.load(f)

In [ ]:
from eval.HitRate import hitrate_sparse_retrieval, hitrate_dense_retrieval, hitrate_hybrid_retrieval
from eval.MeanReciprocalRank import mrr_sparse_retrieval, mrr_dense_retrieval, mrr_hybrid_retrieval
from eval.Precision import precision_sparse_retrieval, precision_dense_retrieval, precision_hybrid_retrieval
from eval.Recall import recall_sparse_retrieval, recall_dense_retrieval, recall_hybrid_retrieval


a = hitrate_sparse_retrieval (sparse_retrieval, dataset)
b = hitrate_dense_retrieval (dense_retrieval, dataset)
#c = hitrate_hybrid_retrieval (sparse_retrieval, dense_retrieval, dataset, 15)

d = mrr_sparse_retrieval (sparse_retrieval, dataset)
e = mrr_dense_retrieval (dense_retrieval, dataset)
#f = mrr_hybrid_retrieval (sparse_retrieval, dense_retrieval, dataset, 15)

g = precision_sparse_retrieval (sparse_retrieval, dataset)
h = precision_dense_retrieval (dense_retrieval, dataset)
#i = precision_hybrid_retrieval (sparse_retrieval, dense_retrieval, dataset, 15)

j = recall_sparse_retrieval (sparse_retrieval, dataset)
k = recall_dense_retrieval (dense_retrieval, dataset)
#l = recall_hybrid_retrieval (sparse_retrieval, dense_retrieval, dataset, 15)




print (f"{a}\n{b}")
print ("---" * 50)

print (f"{d}\n{e}")
print ("---" * 50)

print (f"{g}\n{h}")
print ("---" * 50)

print (f"{j}\n{k}")
print ("---" * 50)

<hr>

<h1> Reranking </h1>

Usando um modelo Cross Encoder

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

path = r"C:\Users\Admin\Desktop\models\Cross Encoder"

cross_encoder_model = AutoModelForSequenceClassification.from_pretrained (path, device_map = "cuda")
cross_encoder_model_tokenization = AutoTokenizer.from_pretrained (path)

In [ ]:

query_reranker = [query] * len (teste)
chunks = [t[2] for t in teste]
docs_names = [t[0] for t in teste]
chunks_ids = [t[1] for t in teste]

pares = list(zip(query_reranker, chunks)) # [query, chunks]
#print (pares)

inputs = cross_encoder_model_tokenization (pares, return_tensors = "pt", padding = True, truncation = True)
#print (inputs)

cross_encoder_model.eval()
with torch.no_grad():
    logits = cross_encoder_model (**inputs).logits
    #print (logits)

#Organiza os logits com os chunks correspondentes #Muita Atenção! Para calcular HitRate e MRR com as funções criadas é necessário estar no mesmo formato do output do RRF.
rerank = sorted(zip(docs_names, chunks_ids, chunks, logits.tolist()), key = lambda x: x[3], reverse = True) #x[3] define que arg é usado pelo reverse

#display (rerank)


<h3> Eval RAG + Reranking </h3>

In [ ]:
from eval.HitRate import hitrate_sparse_system, hitrate_dense_system, hitrate_hybrid_system
from eval.MeanReciprocalRank import mrr_sparse_system, mrr_dense_system, mrr_hybrid_system
from eval.Precision import precision_sparse_system, precision_dense_system, precision_hybrid_system
from eval.Recall import recall_sparse_system, recall_dense_system, recall_hybrid_system


#a = hitrate_sparse_system (sparse_retrieval, dataset, r"C:\Users\Admin\Desktop\models\Cross Encoder", 5)
#b = hitrate_dense_system (dense_retrieval, dataset, r"C:\Users\Admin\Desktop\models\Cross Encoder", 5)
c = hitrate_hybrid_system (sparse_retrieval, dense_retrieval, dataset, r"C:\Users\Admin\Desktop\models\Cross Encoder", 5)

#d = mrr_sparse_system (sparse_retrieval, dataset, r"C:\Users\Admin\Desktop\models\Cross Encoder", 5)
#e = mrr_dense_system (dense_retrieval, dataset, r"C:\Users\Admin\Desktop\models\Cross Encoder", 5)
f = mrr_hybrid_system (sparse_retrieval, dense_retrieval, dataset, r"C:\Users\Admin\Desktop\models\Cross Encoder", 5)

#g = precision_sparse_system (sparse_retrieval, dataset, r"C:\Users\Admin\Desktop\models\Cross Encoder", 5)
#h = precision_dense_system (dense_retrieval, dataset, r"C:\Users\Admin\Desktop\models\Cross Encoder", 5)
i = precision_hybrid_system (sparse_retrieval, dense_retrieval, dataset, r"C:\Users\Admin\Desktop\models\Cross Encoder", 5)

#j = recall_sparse_system (sparse_retrieval, dataset, r"C:\Users\Admin\Desktop\models\Cross Encoder", 5)
#k = recall_dense_system (dense_retrieval, dataset, r"C:\Users\Admin\Desktop\models\Cross Encoder", 5)
l = recall_hybrid_system (sparse_retrieval, dense_retrieval, dataset, r"C:\Users\Admin\Desktop\models\Cross Encoder", 5)


#print (a)
#print (b)
print (c)
print ("----" *50)

#print (d)
#print (e)
print (f)
print ("----" *50)

#print (g)
#print (h)
print (i)
print ("----" *50)

#print (j)
#print (k)
print (l)
print ("----" *50)


<hr>

<h1> Modelo Qwen2.5-0.5B-Instruct </h1>

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

path = r"C:\Users\Admin\Desktop\models\Qwen2.5-0.5B-Instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"

model = AutoModelForCausalLM.from_pretrained (path, device_map = device)
tokenizer2 = AutoTokenizer.from_pretrained (path)

device


In [ ]:
with open (r"C:\Users\Admin\Desktop\ip\How To RAG\eval\dataset_pós_answer.json", "r", encoding = "utf-8") as f:
    dataset_fim = json.load(f)

In [ ]:
dataa = []

for q in dataset_fim:
    #print (q)

    #Selecionar query
    query = q["query"]

    sparse_ret = sparse_retrieval.invoke (query) #Sparse Retrieval
    dense_ret = dense_retrieval.invoke (query) #Dense Retrieval

    rrf = Reciprocal_Rank_Fusion ([sparse_ret, dense_ret]) #Reciprocal Rank Fusion

    #Preparar o input para o Cross Encoder Model que precisa de receber [query, chunk]
    query_reranker = [query] * len (rrf) 
    chunks = [t[2] for t in rrf]

    pares = list(zip(query_reranker, chunks))
    ####

    inputs = cross_encoder_model_tokenization (pares, return_tensors = "pt", padding = True, truncation = True).to(device) #Tokenization input

    cross_encoder_model.eval()
    with torch.no_grad():
        logits = cross_encoder_model (**inputs).logits #logits

    rerank = sorted(zip(chunks, logits.tolist()), key = lambda x: x[1], reverse = True) #Organização do output
    #display (rerank)

    feed_llm = [chunks for chunks, scores in rerank [:5]] #Os 5 melhores..
    #display (feed_llm)

    
    prompt = query

    ##Analisar isto ##Interssante perceber se tokens especiais influenciam a resposta do modelo. ##Prompt Engineering https://www.youtube.com/watch?v=ysPbXH0LpIE&t=958s
    chat_template = f"""
    <|system|>
    És um assistente de IA alimentado por um sistema RAG, que tem como único e principal objetivo responder a perguntas realizadas pelos utilizadores de acordo com o contexto fornecido.

    Deves responder de maneira curta e direta, num tom amigável.
    
    Regras:
        1. É fulcral e obrigatório que respondas apenas consoante o contexto fornecido.
        2. Não uses conhecimento externo.
        3. Se a resposta não existir no contexto, responde exatamente: "Informação não presente no contexto fornecido"
        4. Responde de forma breve e direta.
        5. Formata a resposta de forma clara.

    Tens como principal e única função responder a perguntas de acordo com o Contexto fornecido. Não deves inventar nem aceder a informação externa. 
    Fazes parte de um sistemas RAG e deves apenas responder de acordo com o Contexto fornecido. <|end|>

    <|user|>
    <Contexto>

    {feed_llm}
    
    </Contexto>
    
    Pergunta:
    {prompt} <|end|>

    <|assistant|>

    """
    #print (chat_template)

    
    model_inputs = tokenizer2 (chat_template, return_tensors = "pt").to(device) #Tokenização 

    with torch.no_grad():
        generated_ids = model.generate(**model_inputs, max_new_tokens = 512) #Inferência

    ## Recortar a pergunta da resposta
    input_length = model_inputs["input_ids"].shape[1]
    response_tokens = generated_ids[0][input_length:]
    ##

    output = tokenizer2.decode(response_tokens, skip_special_tokens = True) ##Resposta final
    #print (output)

    dataa.append (query)
    dataa.append (feed_llm)
    dataa.append (output)
    

In [ ]:
import pandas as pd

df = pd.DataFrame (dataa, columns = ["x"])

df.to_csv ("Qwen2.5-0.5B-Instruct - Output Analysis.csv", index = False)
#dataa

<h3> Answer Relevance </h3>

In [ ]:
answer_rel = [1, 0.5, 1, 0.6, 0.2, 0.6, 0.4, 1, 0.4, 1, 1, 0.9, 1, 1, 1, 0.2, 1, 0.8, 0.7, 0.2, 0, 0.9, 0, 0.15, 0.70, 0.90, 0.75] #LLM As A Judge

answer_rel_final = sum(answer_rel) / len(answer_rel)

print (len(answer_rel))
print (sum(answer_rel) / len(answer_rel))

<h3> Faithfulness </h3>


In [ ]:
faith = [1, 0.1, 1, 0.2, 0, 0.3, 0.1, 1, 0.2, 1, 1, 0.8, 1, 1, 0.9, 0.1, 1, 0.9, 0.8, 0, 0, 0.9, 0, 0.85, 0.35, 0.55, 0.40] #LLM As A Judge

faith_final = sum(faith) / len(faith) 

print (len(faith))
print (sum(faith) / len(faith))

<h3> Score Final de RAG + Reranker + Modelo Qwen2.5-0.5B-Instruct </h3>


In [ ]:
retr_score = (0.74 + 0.20 + 0.85 + 0.74) / 4
resposta_score = (0.40 * answer_rel_final) + (0.60 * faith_final)

x = retr_score * 0.60 + resposta_score * 0.40

print (x * 100)

<hr>

<h1> Modelo Phi-3.5-mini-instruct-Q4 </h1>

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

path = r"C:\Users\Admin\Desktop\models\Phi-3.5-mini-instruct-Q4"

device = "cuda" if torch.cuda.is_available() else "cpu"

model = AutoModelForCausalLM.from_pretrained (path, device_map = device)
tokenizer2 = AutoTokenizer.from_pretrained (path)

device

<h3> Answer Relevance </h3>


In [ ]:
answer_rel = [1, 1, 0.70, 0.95, 1, 1, 0.4, 0.2, 0.95, 0.5, 1, 0.95, 1, 1, 0.95, 0.8, 0.85, 0.9, 0.3, 0.9, 0.95, 1, 0.4, 0.2, 0.9, 1, 0.98]

answer_rel_final = sum(answer_rel) / len(answer_rel)

answer_rel_final

<h3> Faithfulness </h3>

In [ ]:
faith = [1, 1, 1, 1, 1, 1, 0.6, 0, 0.95, 1, 1, 0.95, 1, 1, 1, 1, 1, 0.95, 0.2, 0.9, 0.85, 0.95, 0.2, 0.1, 0.75, 0.9, 0.97]

faith_final = sum(faith) / len(faith)

faith_final

<h3> Score Final de RAG + Reranker + Modelo Phi-3.5-mini-instruct-Q4</h3>


In [ ]:
retr_score = (0.74 + 0.20 + 0.85 + 0.74) / 4
resposta_score = (0.40 * answer_rel_final) + (0.60 * faith_final)

x = retr_score * 0.60 + resposta_score * 0.40

print (x * 100)